# Model L — Nepali Pretraining (Kaggle)

25,379,840-parameter decoder-only Transformer, 569,684,621 training tokens,
one pass = **17,429 steps** at 32 x 512 x 2 = 32,768 tokens/step.

Kaggle has no `drive.mount`, so **rclone** does the job instead: it pulls the
packed tokens in at session start and pushes every checkpoint back out to Drive.
That second part matters — `/kaggle/working` is discarded when an interactive
session dies, so an un-mirrored run would lose everything on a disconnect.

### Before running

1. **Settings -> Internet: ON** (required for rclone)
2. **Settings -> Accelerator: GPU T4**
3. **Add-ons -> Secrets -> Add secret**, label `RCLONE_CONF`, and paste the
   output of `rclone config show gdrive:` from your machine — including the
   `[gdrive]` header line.

> The config holds an OAuth refresh token for your Drive. Kaggle Secrets are
> encrypted and scoped to your own notebooks.

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'Settings -> Accelerator -> GPU T4'
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))

## 2. rclone

In [ ]:
import os, re
from kaggle_secrets import UserSecretsClient

!curl -s https://rclone.org/install.sh | bash > /dev/null 2>&1


def normalize_rclone_conf(conf: str) -> str:
    """Restore an rclone config whose newlines were flattened to spaces.

    Kaggle's Secrets UI stores a multi-line value on a single line, but rclone
    needs real line breaks or it reports 'didn't find section in config file'.
    Re-insert a break before each [section] and each key=. Safe because rclone
    writes the OAuth token as compact JSON containing no spaces.
    """
    if len(conf.strip().splitlines()) > 1:
        return conf if conf.endswith('\n') else conf + '\n'
    s = conf.strip().replace('\\n', '\n')
    if len(s.splitlines()) > 1:
        return s + '\n'
    s = re.sub(r'\s*(\[[^\]\s]+\])\s*', r'\n\1\n', s)
    s = re.sub(r'\s+([A-Za-z_][A-Za-z0-9_]*)\s*=\s*', r'\n\1 = ', s)
    return '\n'.join(l.strip() for l in s.splitlines() if l.strip()) + '\n'


conf = normalize_rclone_conf(UserSecretsClient().get_secret('RCLONE_CONF'))

# RCLONE_CONFIG names the file explicitly: Kaggle's $HOME is not reliably /root,
# and the variable is inherited by every subprocess, including the checkpoint
# mirroring inside trainer.py.
CONF_PATH = '/kaggle/working/rclone.conf'
with open(CONF_PATH, 'w') as fh:
    fh.write(conf)
os.environ['RCLONE_CONFIG'] = CONF_PATH

lines = conf.splitlines()
print('lines        :', len(lines))
print('has [gdrive] :', any(l.strip() == '[gdrive]' for l in lines))
print('keys         :', [l.split('=')[0].strip() for l in lines if '=' in l])
assert any(l.strip() == '[gdrive]' for l in lines), 'RCLONE_CONF has no [gdrive] section'

!rclone listremotes
!rclone lsd gdrive:24KaratAu

## 3. Code and data from Drive

In [ ]:
import os
DRIVE = 'gdrive:24KaratAu'

!rclone copy {DRIVE}/code /kaggle/working/phase2 --exclude "__pycache__/**"
%cd /kaggle/working/phase2

# ~1.18 GiB of packed tokens into scratch (not counted against the 20 GB output cap)
!mkdir -p /tmp/tokens
!rclone copy {DRIVE}/nepali/tokens /tmp/tokens -P --stats 15s

for f in sorted(os.listdir('/tmp/tokens')):
    print(f'  {f:24} {os.path.getsize(f"/tmp/tokens/{f}")/2**20:9.1f} MiB')

## 4. Sanity tests

In [ ]:
!pip install -q pytest && python -m pytest tests/ -q

## 5. Smoke run — 100 steps

Throwaway directory, ~1 minute. Verifies the config, memory, and that loss moves
before committing to a multi-hour run.

In [ ]:
!rm -rf /tmp/smoke && mkdir -p /tmp/smoke
!python -m src.train.trainer \
    --config nepali/configs/train.yaml \
    --max-steps 100 \
    --ckpt-dir /tmp/smoke/ckpt \
    --log-jsonl /tmp/smoke/log.jsonl \
    --no-mirror

In [ ]:
import json
rows = [json.loads(l) for l in open('/tmp/smoke/log.jsonl')]
tr = [r for r in rows if 'train_loss' in r]
first, last = tr[0]['train_loss'], tr[-1]['train_loss']
tps = sum(r['tokens_per_sec'] for r in tr) / len(tr)
peak = max(r.get('peak_mem_gib', 0) for r in tr)

print(f'loss {first:.3f} -> {last:.3f}   ({tps:,.0f} tok/s)   peak GPU {peak:.1f} GiB')
assert last < first, 'loss did not decrease -- stop and investigate'
assert last < 8.987, 'loss above uniform-random -- something is wrong'
print(f'projected full run: {17429 * 32768 / tps / 3600:.2f} hours')
print('SMOKE TEST PASSED')

## 6. Full run

~4-5 hours. Every checkpoint is mirrored to Drive as it is written, so a
disconnect costs at most 500 steps. To resume, re-run this notebook from the
top — cell 3 restores the code and data, and the trainer pulls the last
checkpoint back down automatically.

In [ ]:
# Recover any checkpoint from a previous session before starting.
!mkdir -p /kaggle/working/checkpoints /kaggle/working/logs
!rclone copy gdrive:24KaratAu/nepali/checkpoints /kaggle/working/checkpoints
!rclone copy gdrive:24KaratAu/nepali/logs /kaggle/working/logs

!python -m src.train.trainer --config nepali/configs/train.yaml

## 7. Verify on Drive

In [ ]:
!rclone lsl gdrive:24KaratAu/nepali/checkpoints
!rclone copy gdrive:24KaratAu/nepali/logs/train.jsonl /tmp/

import json
rows = [json.loads(l) for l in open('/tmp/train.jsonl')]
va = [r for r in rows if 'val_loss' in r]
tr = [r for r in rows if 'train_loss' in r]
print(f'\n{len(tr)} train points, {len(va)} val points')
if va:
    b = min(va, key=lambda r: r['val_loss'])
    print(f"final train loss {tr[-1]['train_loss']:.4f}")
    print(f"best val loss    {b['val_loss']:.4f} (ppl {b['val_ppl']:.2f}) at step {b['step']:,}")